<span style="color:orange">
 <h2> Databircks-Assignemnt 2 (Basic Data Cleaning Transformation)
</span>


<b> Find the count of duplicate employee records in the input file (based on id)?

In [0]:
# Write your code using the Spark Function

# Read the CSV file
df = spark.read.option("header", True).option("inferSchema", True).csv("/Workspace/Users/miketyson.rocks69@gmail.com/PySpark_Practice_Set/Assignment/EmployeeData.csv")

# Count total records
total_records = df.count()

# Count unique employee IDs
unique_ids = df.select("id").distinct().count()

# Count duplicate records based on id
duplicate_records = total_records - unique_ids

print(f"Total records: {total_records}")
print(f"Unique IDs: {unique_ids}")
print(f"Duplicate records: {duplicate_records}")

In [0]:
%sql
SELECT 
  COUNT(*) AS total_records,
  COUNT(DISTINCT id) AS unique_ids,
  COUNT(*) - COUNT(DISTINCT id) AS duplicate_records
FROM employees

<b> Find out how many records have Gender value missing.

In [0]:
# Write your code using the Spark Function
from pyspark.sql.functions import col, sum, when

df.select(
    sum(
        when(col("Gender").isNull() | (col("Gender") == ""), 1)
        .otherwise(0)
    ).alias("missing_gender_count")
).show()

d
<b> Are there any missing values in the "bonus" field? If so, filled them defualt bonus 100.

In [0]:
# Write your code using the Spark Function
from pyspark.sql.functions import col, count, when

# Count missing bonus values
missing_bonus = df.filter(col("bonus").isNull()).count()
print(f"Records with missing bonus: {missing_bonus}")

# Fill missing bonus with default value 100
df = df.fillna({"bonus": 100})

# Verify no more missing bonus values
remaining_missing = df.filter(col("bonus").isNull()).count()
print(f"Records with missing bonus after fill: {remaining_missing}")

df.select("id", "bonus").show(10)

d
<b> Are there any employees with negative salary or bonus amounts in the input file? If so, how many?

In [0]:
# Write your code using the Spark Function
from pyspark.sql.functions import col

# Count employees with negative salary
negative_salary = df.filter(col("salary") < 0).count()
print(f"Employees with negative salary: {negative_salary}")

# Count employees with negative bonus
negative_bonus = df.filter(col("bonus") < 0).count()
print(f"Employees with negative bonus: {negative_bonus}")

# Count employees with negative salary OR bonus
total_negative = df.filter((col("salary") < 0) | (col("bonus") < 0)).count()
print(f"Total employees with negative salary or bonus: {total_negative}")

if total_negative > 0:
    print("\nSample records with negative salary or bonus:")
    df.filter((col("salary") < 0) | (col("bonus") < 0)).select("id", "first_name", "last_name", "salary", "bonus").show()

d
<b> Replace all the null/emtpy value in email column with admin@diacto.com

In [0]:
# Write your code using the Spark Function
from pyspark.sql.functions import col, when, count

# Count null or empty email values
missing_email = df.filter(col("email").isNull() | (col("email") == "")).count()
print(f"Records with null/empty email: {missing_email}")

# Replace null/empty email with admin@diacto.com
df = df.withColumn("email", when(col("email").isNull() | (col("email") == ""), "admin@diacto.com").otherwise(col("email")))

# Verify no more null/empty email values
remaining_missing = df.filter(col("email").isNull() | (col("email") == "")).count()
print(f"Records with null/empty email after replacement: {remaining_missing}")

df.select("id", "email").show(10)

d
<b> Remove all the records where any record has any null values. Find out the total count of the records now.

In [0]:
# Write your code using the Spark Function

# Count records before dropping nulls
before_count = df.count()
print(f"Total records before dropping nulls: {before_count}")

# Remove all records where any column has a null value
df_clean = df.na.drop()

# Count records after dropping nulls
after_count = df_clean.count()
print(f"Total records after dropping nulls: {after_count}")
print(f"Records removed: {before_count - after_count}")